# LeetCode #86 — Partition List: hai danh sách giả rồi nối lại

**Mức độ:** Medium · **Chủ đề:** Linked List, Two Pointers

Skeleton trên LeetCode:

```python
class Solution:
    def partition(self, head: Optional[ListNode], x: int) -> Optional[ListNode]:
```

Mục tiêu của notebook:

1. Tách danh sách thành **hai** danh sách con bằng hai nút giả (`before`, `after`).
2. Giữ **thứ tự tương đối** (stable partition) — điều mà cách "đổi giá trị" không làm được.
3. Lỗi kinh điển: **quên cắt đuôi** `after_tail.next = None` → tạo **vòng lặp**.
4. Liên hệ với bước phân hoạch của Quicksort và #75 Sort Colors.

# 1. Bài toán là gì?

Cho `head` của một danh sách liên kết và số `x`. Sắp xếp lại sao cho mọi nút có giá trị **nhỏ hơn `x`** đứng trước mọi nút có giá trị **lớn hơn hoặc bằng `x`**.

Phải **giữ nguyên thứ tự tương đối** ban đầu của các nút trong mỗi phần.

**Constraints (theo đề):**

- Số nút trong khoảng `[0, 200]`.
- `-100 <= Node.val <= 100`
- `-200 <= x <= 200`

---

## Ví dụ

```text
head = 1 -> 4 -> 3 -> 2 -> 5 -> 2,  x = 3   ->   1 -> 2 -> 2 -> 4 -> 3 -> 5
head = 2 -> 1,                      x = 2   ->   1 -> 2
```

Phần `< 3`: `1, 2, 2` (đúng thứ tự xuất hiện). Phần `>= 3`: `4, 3, 5` (đúng thứ tự xuất hiện) — **không** sắp xếp lại thành `3, 4, 5`.

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Danh sách liên kết đơn (singly linked list)

Cuối mục có phần **2.2 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Danh sách liên kết đơn (singly linked list)

### Danh sách liên kết là gì?

Mảng (`list` của Python) lưu các phần tử **liền nhau** trong bộ nhớ, nên truy cập `a[i]` tức thì. Danh sách liên kết thì khác: mỗi phần tử là một **nút** (node) riêng lẻ, chứa **giá trị** và **con trỏ** `next` chỉ tới nút kế tiếp. Nút cuối có `next = None`.

```text
head
 │
 ▼
┌───┬───┐   ┌───┬───┐   ┌───┬────┐
│ 1 │ ●─┼──►│ 2 │ ●─┼──►│ 3 │None│
└───┴───┘   └───┴───┘   └───┴────┘
 val next
```

Ta chỉ giữ con trỏ tới nút đầu (`head`). Muốn tới nút thứ `k` phải đi qua `k` bước — **không** có truy cập ngẫu nhiên.

| | Mảng | Danh sách liên kết |
|---|---|---|
| Lấy phần tử thứ `i` | `O(1)` | `O(i)` |
| Chèn/xóa ở đầu | `O(n)` | `O(1)` |
| Chèn/xóa sau một nút đã có trong tay | `O(n)` | `O(1)` |

In [1]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

# Tạo bằng tay: 1 -> 2 -> 3
c = ListNode(3)
b = ListNode(2, c)
a = ListNode(1, b)
head = a
print(head.val, head.next.val, head.next.next.val, head.next.next.next)

1 2 3 None


### Duyệt danh sách và hai hàm tiện ích

Mẫu duyệt cơ bản: con trỏ `cur` bắt đầu ở `head`, mỗi bước `cur = cur.next`, dừng khi `cur is None`.

In [2]:
def tao_danh_sach(values):
    """[1, 2, 3] -> 1 -> 2 -> 3, trả về head."""
    dummy = tail = ListNode()          # nút giả để khỏi xử lý riêng nút đầu
    for v in values:
        tail.next = ListNode(v)
        tail = tail.next
    return dummy.next

def ra_list(head):
    out, cur = [], head
    while cur:                          # duyệt tới khi hết
        out.append(cur.val)
        cur = cur.next
    return out

h = tao_danh_sach([4, 5, 6, 7])
print("ra_list:", ra_list(h))
dem, cur = 0, h
while cur:
    dem += 1
    cur = cur.next
print("độ dài:", dem)

ra_list: [4, 5, 6, 7]
độ dài: 4


### Chèn và xóa: chỉ cần đổi con trỏ

Chèn nút `x` **sau** nút `p`: `x.next = p.next; p.next = x` (thứ tự quan trọng — làm ngược sẽ mất phần sau).

Xóa nút **sau** `p`: `p.next = p.next.next`.

```text
Chèn x sau p:   p ──► q        ⟹   p ──► x ──► q
Xóa sau p:      p ──► q ──► r  ⟹   p ──────────► r
```

Muốn xóa một nút, ta cần nút **đứng trước** nó. Nếu nút cần xóa là `head` thì không có nút trước → dùng **nút giả (dummy)** đặt trước `head`.

In [3]:
h = tao_danh_sach([1, 2, 4])
p = h.next                              # nút 2
x = ListNode(3)
x.next = p.next                         # 3 -> 4
p.next = x                              # 2 -> 3
print("sau khi chèn 3:", ra_list(h))

def xoa_gia_tri(head, target):
    dummy = ListNode(0, head)           # dummy đứng trước head
    prev = dummy
    while prev.next:
        if prev.next.val == target:
            prev.next = prev.next.next  # bỏ qua nút cần xóa
        else:
            prev = prev.next
    return dummy.next                   # head mới (có thể đã đổi)

print("xóa mọi số 1:", ra_list(xoa_gia_tri(tao_danh_sach([1, 1, 2, 1, 3]), 1)))

sau khi chèn 3: [1, 2, 3, 4]
xóa mọi số 1: [2, 3]


### Đảo ngược danh sách — kỹ thuật ba con trỏ

Đi dọc danh sách, lật mũi tên của từng nút cho trỏ về nút **trước** nó. Cần lưu `nxt` trước khi lật, vì sau khi lật ta mất đường đi tiếp.

```text
prev  cur   nxt
None   1 ──► 2 ──► 3          bước 1: 1.next = None
None ◄─1     2 ──► 3          bước 2: 2.next = 1
None ◄─1 ◄─ 2     3          bước 3: 3.next = 2  → head mới = 3
```

In [4]:
def dao_nguoc(head):
    prev, cur = None, head
    while cur:
        nxt = cur.next      # 1) lưu đường đi tiếp
        cur.next = prev     # 2) lật mũi tên
        prev = cur          # 3) tiến prev
        cur = nxt           # 4) tiến cur
    return prev             # prev là head mới

print("đảo [1..5]:", ra_list(dao_nguoc(tao_danh_sach([1, 2, 3, 4, 5]))))

đảo [1..5]: [5, 4, 3, 2, 1]


### Con trỏ nhanh / chậm (fast & slow)

`slow` đi 1 bước, `fast` đi 2 bước mỗi vòng.

- Khi `fast` tới cuối, `slow` ở **giữa** danh sách.
- Nếu danh sách có **chu trình**, `fast` sẽ đuổi kịp `slow` (thuật toán Floyd).

In [5]:
def nut_giua(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    return slow

print("giữa [1..5]:", nut_giua(tao_danh_sach([1, 2, 3, 4, 5])).val,
      "| giữa [1..6]:", nut_giua(tao_danh_sach([1, 2, 3, 4, 5, 6])).val, "(nút giữa thứ hai)")

def co_chu_trinh(head):
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
        if slow is fast:
            return True
    return False

v = tao_danh_sach([1, 2, 3, 4])
v.next.next.next.next = v.next          # 4 -> 2 tạo chu trình
print("có chu trình?", co_chu_trinh(v), "| danh sách thường:", co_chu_trinh(tao_danh_sach([1, 2])))

giữa [1..5]: 3 | giữa [1..6]: 4 (nút giữa thứ hai)
có chu trình? True | danh sách thường: False


### Lỗi hay gặp với danh sách liên kết

- Truy cập `cur.next.val` khi `cur.next` là `None` → `AttributeError`. Luôn kiểm tra `cur and cur.next`.
- Đổi `cur.next` **trước** khi lưu nút kế → mất phần còn lại của danh sách.
- Quên trường hợp danh sách rỗng (`head is None`) hoặc chỉ một nút.
- Quên dùng dummy khi `head` có thể bị xóa/thay → phải viết nhánh riêng, dễ sai.
- In danh sách có chu trình bằng vòng `while cur` → lặp vô hạn.

## 2.2 Áp dụng vào bài này

Chia **danh sách liên kết** thành hai danh sách (nhỏ hơn `x` / còn lại), mỗi danh sách có **nút giả** riêng để nối đuôi dễ dàng, rồi nối hai danh sách lại. Nhớ **cắt đuôi** (`tail.next = None`) để không tạo chu trình.

# 3. Nền tảng

In [6]:
from typing import Optional, List
import random


class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next


def build_list(values: List[int]) -> Optional[ListNode]:
    dummy = ListNode()
    tail = dummy
    for v in values:
        tail.next = ListNode(v)
        tail = tail.next
    return dummy.next


def to_list(head: Optional[ListNode], limit: int = 1000) -> List[int]:
    out = []
    while head and len(out) < limit:          # limit: tránh treo nếu danh sách có vòng
        out.append(head.val)
        head = head.next
    return out

print(to_list(build_list([1, 4, 3, 2, 5, 2])))

[1, 4, 3, 2, 5, 2]


`to_list` có tham số `limit` để **không bị treo** khi gặp danh sách có vòng — ta sẽ cần nó ở phần "lỗi dễ mắc".

# 4. Baseline: gom giá trị vào hai list Python

`O(n)` bộ nhớ phụ và tạo **nút mới** — đúng kết quả nhưng không "sắp xếp lại các nút" như tinh thần của đề.

In [7]:
def partition_values(head: Optional[ListNode], x: int) -> Optional[ListNode]:
    values = to_list(head)
    return build_list([v for v in values if v < x] + [v for v in values if v >= x])

print(to_list(partition_values(build_list([1, 4, 3, 2, 5, 2]), 3)))

[1, 2, 2, 4, 3, 5]


# 5. Ý tưởng chính: hai danh sách giả

Duyệt một lượt, **gắn lại** từng nút vào cuối một trong hai danh sách:

- `before`: các nút `< x`
- `after`: các nút `>= x`

Mỗi danh sách có một nút giả (dummy) làm đầu và một con trỏ `tail` ở cuối. Cuối cùng:

```text
after_tail.next = None          # cắt đuôi cũ (rất quan trọng!)
before_tail.next = after.next   # nối hai phần
return before.next
```

Vì gắn theo thứ tự duyệt, thứ tự tương đối trong mỗi phần được giữ nguyên.

In [8]:
def partition_two_lists(head: Optional[ListNode], x: int) -> Optional[ListNode]:
    before = before_tail = ListNode()
    after = after_tail = ListNode()
    node = head
    while node:
        if node.val < x:
            before_tail.next = node
            before_tail = node
        else:
            after_tail.next = node
            after_tail = node
        node = node.next
    after_tail.next = None
    before_tail.next = after.next
    return before.next

print(to_list(partition_two_lists(build_list([1, 4, 3, 2, 5, 2]), 3)))
print(to_list(partition_two_lists(build_list([2, 1]), 2)))

[1, 2, 2, 4, 3, 5]
[1, 2]


`before = before_tail = ListNode()` gán **cùng một** nút cho cả hai tên: ban đầu danh sách rỗng nên đầu và đuôi là nút giả.

# 6. Mô phỏng

In [9]:
def trace_partition(values: List[int], x: int) -> List[int]:
    head = build_list(values)
    before = before_tail = ListNode("B")
    after = after_tail = ListNode("A")
    node = head
    print(f"x = {x}")
    while node:
        nxt = node.next
        if node.val < x:
            before_tail.next = node
            before_tail = node
            where = "before"
        else:
            after_tail.next = node
            after_tail = node
            where = "after "
        node.next = None                     # chỉ để in cho dễ nhìn
        print(f"  nút {node.val} -> {where} | before: {to_list(before.next)}  after: {to_list(after.next)}")
        node = nxt
    after_tail.next = None
    before_tail.next = after.next
    print("Nối lại:", to_list(before.next))
    return to_list(before.next)

_ = trace_partition([1, 4, 3, 2, 5, 2], 3)

x = 3
  nút 1 -> before | before: [1]  after: []
  nút 4 -> after  | before: [1]  after: [4]
  nút 3 -> after  | before: [1]  after: [4, 3]
  nút 2 -> before | before: [1, 2]  after: [4, 3]
  nút 5 -> after  | before: [1, 2]  after: [4, 3, 5]
  nút 2 -> before | before: [1, 2, 2]  after: [4, 3, 5]
Nối lại: [1, 2, 2, 4, 3, 5]


(Ở hàm `trace_partition` ta cắt `node.next = None` mỗi bước để in hai danh sách con rõ ràng; trong lời giải chính không cần — chỉ cần cắt đuôi `after` ở cuối.)

# 7. Phiên bản đúng format LeetCode

In [10]:
class Solution:
    def partition(self, head: Optional[ListNode], x: int) -> Optional[ListNode]:
        before = before_tail = ListNode()
        after = after_tail = ListNode()
        while head:
            if head.val < x:
                before_tail.next = head
                before_tail = head
            else:
                after_tail.next = head
                after_tail = head
            head = head.next
        after_tail.next = None
        before_tail.next = after.next
        return before.next

In [11]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<22} PASS  {actual}")


def run(values, x):
    head = build_list(values)
    nodes = []
    node = head
    while node:
        nodes.append(node)
        node = node.next
    out = Solution().partition(head, x)
    res, seen, node = [], set(), out
    while node:
        assert id(node) not in seen, "danh sách có vòng!"
        seen.add(id(node))
        res.append(node.val)
        node = node.next
    assert seen == {id(n) for n in nodes}, "phải dùng lại đúng các nút cũ"
    return res


check("Ví dụ 1", run([1, 4, 3, 2, 5, 2], 3), [1, 2, 2, 4, 3, 5])
check("Ví dụ 2", run([2, 1], 2), [1, 2])
check("Rỗng", run([], 0), [])
check("Toàn nhỏ hơn x", run([1, 2, 3], 10), [1, 2, 3])
check("Toàn lớn hơn x", run([5, 6, 7], 0), [5, 6, 7])
check("Bằng x", run([3, 3, 1], 3), [1, 3, 3])
check("Số âm", run([-1, 5, -3, 0], 0), [-1, -3, 5, 0])

random.seed(86)
for _ in range(5000):
    values = [random.randint(-5, 5) for _ in range(random.randint(0, 12))]
    x = random.randint(-6, 6)
    assert run(values, x) == to_list(partition_values(build_list(values), x)), (values, x)
print("5000 danh sách ngẫu nhiên: đúng thứ tự, không vòng, dùng lại các nút cũ")

Ví dụ 1                PASS  [1, 2, 2, 4, 3, 5]


Ví dụ 2                PASS  [1, 2]
Rỗng                   PASS  []
Toàn nhỏ hơn x         PASS  [1, 2, 3]
Toàn lớn hơn x         PASS  [5, 6, 7]
Bằng x                 PASS  [1, 3, 3]
Số âm                  PASS  [-1, -3, 5, 0]


5000 danh sách ngẫu nhiên: đúng thứ tự, không vòng, dùng lại các nút cũ


Hàm `run` kiểm tra thêm hai điều mà so sánh giá trị không thấy được: kết quả **không có vòng** và dùng **đúng các nút cũ** (không tạo nút mới).

# 8. Vì sao đúng? Độ phức tạp?

- Mỗi nút được gắn vào đúng một trong hai danh sách, theo thứ tự duyệt → mỗi phần giữ thứ tự tương đối.
- Nối `before` rồi `after` → mọi nút `< x` đứng trước mọi nút `>= x`.
- `after_tail.next = None` đảm bảo danh sách kết thúc đúng chỗ.

```text
Time:  O(n)
Space: O(1) — chỉ thêm hai nút giả
```

# 9. Những lỗi dễ mắc

## Lỗi 1: Quên `after_tail.next = None` → vòng lặp

Nút cuối của phần `after` có thể vẫn trỏ tới một nút cũ (thuộc phần `before`) → tạo **vòng**.

In [12]:
def wrong_no_cut(head: Optional[ListNode], x: int) -> Optional[ListNode]:
    before = before_tail = ListNode()
    after = after_tail = ListNode()
    while head:
        if head.val < x:
            before_tail.next = head
            before_tail = head
        else:
            after_tail.next = head
            after_tail = head
        head = head.next
    before_tail.next = after.next                  # SAI: thiếu after_tail.next = None
    return before.next

out = wrong_no_cut(build_list([1, 4, 3, 2, 5, 2]), 3)
print("Sai (in tối đa 15 nút):", to_list(out, limit=15))

Sai (in tối đa 15 nút): [1, 2, 2, 4, 3, 5, 2, 4, 3, 5, 2, 4, 3, 5, 2]


Nút `5` (cuối phần `after`) vẫn trỏ tới nút `2` cuối cùng (thuộc `before`), mà nút `2` đó lại trỏ tới `4` → vòng `2 → 4 → 3 → 5 → 2 → …`. Duyệt bằng `while node` sẽ **chạy mãi**.

## Lỗi 2: Đổi chỗ **giá trị** kiểu Quicksort

Phân hoạch kiểu Quicksort (hai con trỏ, đổi chỗ) **không ổn định** — làm mất thứ tự tương đối.

In [13]:
def wrong_swap_values(head: Optional[ListNode], x: int) -> Optional[ListNode]:
    boundary = head                                 # nút đầu tiên của vùng >= x
    node = head
    while node:
        if node.val < x:
            boundary.val, node.val = node.val, boundary.val   # SAI: không ổn định
            boundary = boundary.next
        node = node.next
    return head

print("Sai :", to_list(wrong_swap_values(build_list([1, 4, 3, 2, 5, 2]), 3)), "| Đúng:", run([1, 4, 3, 2, 5, 2], 3))

Sai : [1, 2, 2, 4, 5, 3] | Đúng: [1, 2, 2, 4, 3, 5]


Phần `>= 3` thành `4, 5, 3` thay vì `4, 3, 5` — thứ tự tương đối bị xáo trộn.

## Lỗi 3: Trả `before.next` khi phần `before` rỗng

Không sai! Khi không có nút nào `< x`, `before_tail` vẫn là nút giả, nên `before_tail.next = after.next` → `before.next` chính là đầu phần `after`. Đây là cái lợi của nút giả.

## Lỗi 4: So sánh `<=` thay vì `<`

Đề: phần đầu là `< x`; các nút **bằng** `x` thuộc phần sau.

# 10. Mẹo nhận diện pattern

**Nhiều danh sách giả** khi cần **tách** một danh sách liên kết thành vài nhóm, giữ thứ tự:

- Mỗi nhóm một cặp `(dummy, tail)`.
- Cuối cùng cắt đuôi, rồi nối các nhóm.

Bài liên quan: **#328 Odd Even Linked List** (tách theo vị trí chẵn/lẻ), **#725 Split Linked List in Parts**, **#2181 Merge Nodes in Between Zeros**, **#75 Sort Colors** (phân hoạch trên mảng, không ổn định).

# 11. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Phân hoạch `3 -> 1 -> 4 -> 1 -> 5 -> 9 -> 2 -> 6` với `x = 4`.

## Bài 2 — Điền chỗ trống

In [14]:
def exercise_2_partition(head: Optional[ListNode], x: int) -> Optional[ListNode]:
    before = before_tail = ListNode()
    after = after_tail = ListNode()
    while head:
        # TODO: gắn head vào before hoặc after
        head = head.next
    # TODO: cắt đuôi after, nối hai phần
    return before.next


# assert to_list(exercise_2_partition(build_list([1, 4, 3, 2, 5, 2]), 3)) == [1, 2, 2, 4, 3, 5]
# assert to_list(exercise_2_partition(build_list([2, 1]), 2)) == [1, 2]
# assert to_list(exercise_2_partition(build_list([]), 1)) == []

## Bài 3 — Biến thể: Odd Even Linked List (LeetCode #328)

Gom các nút ở **vị trí lẻ** (1, 3, 5, …) lên trước, rồi các nút ở **vị trí chẵn**. Giữ thứ tự. `O(1)` bộ nhớ.

```python
odd_even(1 -> 2 -> 3 -> 4 -> 5)       -> 1 -> 3 -> 5 -> 2 -> 4
odd_even(2 -> 1 -> 3 -> 5 -> 6 -> 4 -> 7) -> 2 -> 3 -> 6 -> 7 -> 1 -> 5 -> 4
```

In [15]:
def exercise_3_odd_even(head: Optional[ListNode]) -> Optional[ListNode]:
    # TODO
    pass


# assert to_list(exercise_3_odd_even(build_list([1, 2, 3, 4, 5]))) == [1, 3, 5, 2, 4]
# assert to_list(exercise_3_odd_even(build_list([2, 1, 3, 5, 6, 4, 7]))) == [2, 3, 6, 7, 1, 5, 4]
# assert to_list(exercise_3_odd_even(build_list([]))) == []

## Bài 4 — Biến thể: phân hoạch ba nhóm `< x`, `== x`, `> x` (ổn định)

```python
partition3(1 -> 4 -> 3 -> 2 -> 5 -> 3, 3) -> 1 -> 2 -> 3 -> 3 -> 4 -> 5
```

In [16]:
def exercise_4_partition3(head: Optional[ListNode], x: int) -> Optional[ListNode]:
    # TODO
    pass


# assert to_list(exercise_4_partition3(build_list([1, 4, 3, 2, 5, 3]), 3)) == [1, 2, 3, 3, 4, 5]
# assert to_list(exercise_4_partition3(build_list([5, 1, 5]), 5)) == [1, 5, 5]
# assert to_list(exercise_4_partition3(build_list([]), 0)) == []

## Bài 5 — Mini challenge: Quicksort trên danh sách liên kết

Dùng Bài 4 để viết Quicksort **ổn định** trên danh sách liên kết: lấy giá trị nút đầu làm pivot, phân hoạch ba nhóm, đệ quy sắp xếp nhóm `<` và `>`, rồi nối.

In [17]:
def exercise_5_quicksort_list(head: Optional[ListNode]) -> Optional[ListNode]:
    # TODO
    pass


# assert to_list(exercise_5_quicksort_list(build_list([4, 2, 1, 3]))) == [1, 2, 3, 4]
# assert to_list(exercise_5_quicksort_list(build_list([-1, 5, 3, 4, 0, 3]))) == [-1, 0, 3, 3, 4, 5]
# assert to_list(exercise_5_quicksort_list(build_list([]))) == []

# 12. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_partition` sau khi tự làm.

**Bài 3**
- Mức 1: `odd = head`, `even_head = even = head.next`.
- Mức 2: `while even and even.next`: `odd.next = even.next; odd = odd.next; even.next = odd.next; even = even.next`.
- Mức 3: cuối cùng `odd.next = even_head`.

**Bài 4**
- Mức 1: ba cặp `(dummy, tail)`; cắt đuôi cả ba; nối `less → equal → greater`.
- Mức 2: nối cẩn thận khi có nhóm rỗng — dùng một nút giả cho kết quả và nối lần lượt.

**Bài 5**
- Mức 1: tách ba nhóm (trả cả ba đầu danh sách), đệ quy cho nhóm `<` và `>`.
- Mức 2: nối: tìm đuôi của từng phần rồi gắn.

# 13. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [18]:
_ = trace_partition([3, 1, 4, 1, 5, 9, 2, 6], 4)

x = 4
  nút 3 -> before | before: [3]  after: []
  nút 1 -> before | before: [3, 1]  after: []
  nút 4 -> after  | before: [3, 1]  after: [4]
  nút 1 -> before | before: [3, 1, 1]  after: [4]
  nút 5 -> after  | before: [3, 1, 1]  after: [4, 5]
  nút 9 -> after  | before: [3, 1, 1]  after: [4, 5, 9]
  nút 2 -> before | before: [3, 1, 1, 2]  after: [4, 5, 9]
  nút 6 -> after  | before: [3, 1, 1, 2]  after: [4, 5, 9, 6]
Nối lại: [3, 1, 1, 2, 4, 5, 9, 6]


## Đáp án Bài 2

In [19]:
def answer_2_partition(head: Optional[ListNode], x: int) -> Optional[ListNode]:
    return partition_two_lists(head, x)

assert to_list(answer_2_partition(build_list([1, 4, 3, 2, 5, 2]), 3)) == [1, 2, 2, 4, 3, 5]
assert to_list(answer_2_partition(build_list([2, 1]), 2)) == [1, 2]
assert to_list(answer_2_partition(build_list([]), 1)) == []
print("Bài 2 PASS (giống partition_two_lists ở mục 5)")

Bài 2 PASS (giống partition_two_lists ở mục 5)


## Đáp án Bài 3

In [20]:
def answer_3_odd_even(head: Optional[ListNode]) -> Optional[ListNode]:
    if not head:
        return head
    odd = head
    even_head = even = head.next
    while even and even.next:
        odd.next = even.next
        odd = odd.next
        even.next = odd.next
        even = even.next
    odd.next = even_head
    return head

assert to_list(answer_3_odd_even(build_list([1, 2, 3, 4, 5]))) == [1, 3, 5, 2, 4]
assert to_list(answer_3_odd_even(build_list([2, 1, 3, 5, 6, 4, 7]))) == [2, 3, 6, 7, 1, 5, 4]
assert to_list(answer_3_odd_even(build_list([]))) == []
for n in range(0, 10):
    vals = list(range(n))
    assert to_list(answer_3_odd_even(build_list(vals))) == vals[0::2] + vals[1::2]
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [21]:
def split3(head: Optional[ListNode], x: int):
    heads = [ListNode(), ListNode(), ListNode()]          # < x, == x, > x
    tails = heads[:]
    while head:
        g = 0 if head.val < x else (1 if head.val == x else 2)
        tails[g].next = head
        tails[g] = head
        head = head.next
    for t in tails:
        t.next = None
    return [h.next for h in heads]


def concat(*parts):
    dummy = tail = ListNode()
    for p in parts:
        tail.next = p
        while tail.next:
            tail = tail.next
    return dummy.next


def answer_4_partition3(head: Optional[ListNode], x: int) -> Optional[ListNode]:
    return concat(*split3(head, x))

assert to_list(answer_4_partition3(build_list([1, 4, 3, 2, 5, 3]), 3)) == [1, 2, 3, 3, 4, 5]
assert to_list(answer_4_partition3(build_list([5, 1, 5]), 5)) == [1, 5, 5]
assert to_list(answer_4_partition3(build_list([]), 0)) == []
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [22]:
def answer_5_quicksort_list(head: Optional[ListNode]) -> Optional[ListNode]:
    if not head or not head.next:
        return head
    less, equal, greater = split3(head, head.val)
    return concat(answer_5_quicksort_list(less), equal, answer_5_quicksort_list(greater))

assert to_list(answer_5_quicksort_list(build_list([4, 2, 1, 3]))) == [1, 2, 3, 4]
assert to_list(answer_5_quicksort_list(build_list([-1, 5, 3, 4, 0, 3]))) == [-1, 0, 3, 3, 4, 5]
assert to_list(answer_5_quicksort_list(build_list([]))) == []
for _ in range(2000):
    values = [random.randint(-5, 5) for _ in range(random.randint(0, 15))]
    assert to_list(answer_5_quicksort_list(build_list(values))) == sorted(values)
print("Bài 5 PASS")

Bài 5 PASS


(`concat` đi tới cuối từng phần nên tốn thêm thời gian tuyến tính mỗi tầng; trung bình vẫn `O(n log n)`. Để sắp xếp danh sách liên kết trong thực tế, **Merge Sort** — #148 Sort List — là lựa chọn chuẩn vì luôn `O(n log n)`.)

# 14. Tổng kết

> Hai danh sách giả `before` / `after`, gắn từng nút vào đúng nhóm theo thứ tự duyệt, **cắt đuôi** `after_tail.next = None`, rồi nối `before_tail.next = after.next`. Ổn định, `O(n)` thời gian, `O(1)` bộ nhớ.

---

## Checklist trước khi nộp

- [ ] Có `after_tail.next = None`?
- [ ] So sánh `< x` (nút bằng `x` vào nhóm sau)?
- [ ] Gắn lại nút cũ, không đổi giá trị (giữ ổn định)?
- [ ] Trả `before.next`?